# Your Own Nowcast

In this activity you build a nowcast from start to finish, with your own choices:

- the variables that go into the model;
- the model itself: the lines that create it and fit it.

The data cleaning and the nowcast loop are the ones of Activity 9. The notebook runs as it is, with OLS
and a default list of variables. Change them where it says **COMPLETE**.

## The config loader

The same cell as in Activities 9 and 10: it reads `config.py` and imports the helper functions from
`nwcst_helpers.py`.

In [ ]:
# ===========================================================================
# CONFIG LOADER: edit config.py, NOT this cell.
# Gives you `cfg` (every country knob) and `P` (the standard paths).
# ===========================================================================
import os, sys, importlib, importlib.util

# config.py and nwcst_helpers.py live in activity/, beside this notebook.
# From activity/ that is simply the current folder; from sols/ it is next door.
CODE_DIR = os.getcwd()
if not os.path.exists(os.path.join(CODE_DIR, "config.py")):
    CODE_DIR = os.path.abspath(os.path.join(CODE_DIR, "..", "activity"))

_spec = importlib.util.spec_from_file_location("config", os.path.join(CODE_DIR, "config.py"))
cfg = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(cfg)
P = cfg.derive_paths(CODE_DIR)     # creates raw/, data/, output/, logs/ if missing

# ---------------------------------------------------------------------------
# The model library. joblib/loky workers re-import the module, so its folder
# must be importable, hence the sys.path insert.
# ---------------------------------------------------------------------------
for _p in (os.getcwd(), CODE_DIR):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import nwcst_helpers
importlib.reload(nwcst_helpers)    # pick up edits without restarting the kernel
from nwcst_helpers import *        # deduplicate_correlated, gen_lagged_data, flatten_data,
                                   # mean_fill_dataset, quarter_rows, lagged_target_v

print(f"[CONFIG] {cfg.COUNTRY} ({cfg.CODE}) | "
      f"train {cfg.TRAIN_START} -> test {cfg.TEST_START} -> target {cfg.DESIRED_DATE}")

## Libraries and settings

The models you may want to try are imported here. If your method is not in the list, add its import.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.seasonal import seasonal_decompose

# Models: use any of these in Part 3
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.base import clone

train_start_date = pd.to_datetime(cfg.TRAIN_START)
test_start_date  = pd.to_datetime(cfg.TEST_START)
desired_date     = pd.to_datetime(cfg.DESIRED_DATE)   # the quarter to nowcast (2026Q3)

target_variable = cfg.GDP_TICKER   # real GDP
cpi             = cfg.CPI_TICKER   # CPI, used to deflate nominal series
lags = list(cfg.LAGS)              # vintages: -2 (two months before the quarter ends) ... +2

print(f"target {target_variable} | test from {cfg.TEST_START} | vintages {lags}")

# Part 1: Clean the data

The steps of Activity 9: deflate, remove seasonality, growth rates, then drop the series that are too
short, stopped before 2024, nearly identical to another one, or published too late.

In [ ]:
data = pd.read_csv(os.path.join(P["data"], cfg.DATA_CSV), parse_dates=True, index_col='date')
test_end_date = data[target_variable].dropna().index.max()   # last quarter with GDP

metadata = pd.read_csv(os.path.join(P["data"], cfg.METADATA_CSV))
meta_col = 'ticket'
print(f"{data.shape[1]} series, {data.index.min():%Y-%m} to {data.index.max():%Y-%m}; "
      f"GDP until {test_end_date:%Y-%m}")

Deflate the nominal series with the CPI.

In [ ]:
# =========================================================
# Deflate nominal variables using CPI (base year: cfg.CPI_BASE_YEAR average, CBTT, Jan 2015 = 100)
# =========================================================
# Many series are nominal; deflating them puts everything in real terms.
# We compute a CPI base as the average CPI level in the base year.
base = data[cpi][data.index.year == cfg.CPI_BASE_YEAR].mean()
# data.index.year == 2019 is a boolean mask; it selects only rows where the year is 2019

# metadata.loc[condition][column].values extracts matching column values as a numpy array
nominal = metadata.loc[metadata["nominal"] == 1][meta_col].values

# Deflate: Real_t = Nominal_t / CPI_t * CPI_base   (corrected: was multiplying)
for column in data.columns:       # loop over every column name in the DataFrame
    if column in nominal:         # 'in' checks whether a value exists in a list/array
        data[column] = data[column] / data[cpi] * base    # beta11: divide (deflate) not multiply

# .drop() removes columns (axis=1) or rows (axis=0); inplace=True modifies the DataFrame directly
data.drop(columns=[cpi], inplace=True)
# Drop GDP components
# Drop those columns
data = data.drop(columns=[col for col in data.columns if 'RGDP' in col and col != 'RGDP0000'])
# list comprehension: builds a list of column names where 'RGDP' appears but keeping the target

Remove the seasonal component of the series that are not already seasonally adjusted.

In [ ]:
# .str accessor lets you use string methods on a Series; .str.lower() converts to lowercase
quarterly  = metadata.loc[metadata["freq"].str.lower().str.contains("q")][meta_col].values
monthly    = metadata.loc[metadata["freq"].str.lower().str.contains("m")][meta_col].values
already_sa = metadata.loc[metadata["sa"]==1][meta_col].values   # sa=1 means already seasonally adjusted

# Keep only columns with at least 24 non-NaN values; .notna().sum() counts non-missing entries per column
data = data.loc[:, data.notna().sum() >= 24]

for column in data.columns:
    if column in already_sa:
        continue   # 'continue' skips the rest of the loop body and moves to the next column
    series = data[column].dropna()   # remove NaN before decomposing (seasonal_decompose requires it)
    if column in quarterly:
        # period=4: quarterly series has 4 seasons per year
        decomp = seasonal_decompose(series, model='additive', extrapolate_trend='freq', period=4)
        data[column] = data[column] - decomp.seasonal   # subtract seasonal component to remove seasonality
    else :
        # period=12: monthly series has 12 seasons per year
        decomp = seasonal_decompose(series, model='additive', extrapolate_trend='freq', period=12)
        data[column] = data[column] - decomp.seasonal

Growth rates, then drop the series that are mostly missing or that stopped before 2024.

In [ ]:
# =========================================================
# Convert levels to growth rates (percent change)
# =========================================================
# Quarterly series: quarter-over-quarter growth rate
# All other series: default percent change at their observed frequency
for column in data.columns:
    if column in quarterly:
        # Get non-NaN values with their original index
        non_nan = data[column].dropna()
        
        # .pct_change(periods=1) computes (x_t - x_{t-1}) / x_{t-1}; * 100 gives percentage
        pct_change_values = non_nan.pct_change(periods=1, fill_method=None) * 100
        
        # Create a full-length Series of NaN, then fill in only the computed positions
        result = pd.Series(np.nan, index=data.index)
        result.loc[pct_change_values.index] = pct_change_values   # .loc with an index selects by label
        
        data[column] = result
    else:
        # Store original NaN mask
        original_was_nan = data[column].isna()   # boolean Series: True where value is missing
        
        # For monthly/weekly/daily, calculate normally
        pct_change_values = data[column].pct_change(periods=1, fill_method=None) * 100
        
        # Mask out cases where either current or previous value was originally NaN
        prev_was_nan = data[column].shift(1).isna()   # .shift(1) shifts values down by 1 row
        pct_change_values[prev_was_nan | original_was_nan] = np.nan   # | is the bitwise OR for booleans

        data[column] = pct_change_values

# np.inf arises from pct_change when dividing by zero; replace with NaN to keep data clean
data.replace([np.inf, -np.inf], np.nan, inplace=True)

# =========================================================
# Drop columns in quarterly with more than 75% missing values
# =========================================================
for column in [ col for col in data.columns if col not in quarterly ] :
    # compute fraction of missing values in the training period only
    missing_pct = data[data.index<=test_start_date][column].isna().sum() / len(data[data.index<=test_start_date][column])
    if missing_pct > 0.75:
        data.drop(column, axis=1, inplace=True)

# =========================================================
# Final cleaning
# =========================================================
# Replace infinities created by pct_change with NaN
data.replace([np.inf, -np.inf], np.nan, inplace=True)

# Drop CPI from predictor set (used only for deflation)
#data.drop(cpi, axis=1, inplace=True)

# ~ negates a boolean Series; data[condition] filters rows where condition is True
data = data[~(data.index < train_start_date)].reset_index(drop=False)
# .reset_index(drop=False) moves the date index back to a regular column
data = data.set_index("date").dropna(how="all", axis=0)   # drop rows that are entirely NaN

# pd.to_numeric(..., errors='coerce') converts to number; non-numeric values become NaN
metadata["months_lag"] = (
    pd.to_numeric(metadata["months_lag"], errors="coerce")
      .fillna(1)      # replace any NaN lags with 1 (safe default)
      .astype(int)    # .astype(int) casts the Series to integer dtype
)

# Drop those columns
data = data.drop(columns=[col for col in data.columns if 'RGDP' in col and col != 'RGDP0000'])
for col in data.columns :
    last = data[col].dropna().index.max().year   # find the most recent year with data for this column
    if last < 2024 :
        data.drop(col, axis=1, inplace=True)   # remove columns that stopped updating before 2024

Drop near-duplicates (correlation above 0.98), series published more than 5 months late, and the
index of economic activity (a near copy of GDP).

In [ ]:
data = deduplicate_correlated(data, threshold=0.98,
                              target=target_variable,
                              metadata=metadata, meta_col=meta_col)

# Drop high-publication-lag series: anything published more than 5 months late is
# useless for a nowcast, however well it correlates.
tickets_to_keep = metadata[metadata.months_lag <= 5]['ticket'].tolist()

# Keep only these columns (that exist in data)
data = data[[ col for col in data.columns if col in tickets_to_keep or "RGDP" in col ]]

# Drop the CBTT index of economic activity (RIEA*). It is published about 3 months after the
# quarter and is almost GDP itself, so LARS would pick it first every time. We want indicators
# of GDP, not a copy of it.
data = data.drop(columns=[c for c in data.columns if c.startswith("RIEA")])

print(f"{data.shape[1] - 1} candidate series after cleaning")

# Part 2: Choose your variables

`find_series()` searches the cleaned data by word. It shows each series' ticket, description, frequency,
publication lag (months after the month it describes) and last observation.

In [ ]:
last_obs = data.apply(pd.Series.last_valid_index).dropna()   # series with at least one value
catalog = metadata.loc[metadata.ticket.isin(last_obs.index) & (metadata.ticket != target_variable),
                       ["ticket", "description", "freq", "months_lag"]].copy()
catalog["last_obs"] = pd.to_datetime(catalog.ticket.map(last_obs)).dt.strftime("%Y-%m")
catalog = catalog.set_index("ticket")


def find_series(word):
    """Series of the cleaned data whose description contains `word` (any case)."""
    return catalog[catalog.description.str.contains(word, case=False, na=False)]


find_series("cement")

Search other topics, for example `find_series("credit")`, `find_series("oil")` or `find_series("NTL")`.
The full list is `catalog`.

In [ ]:
find_series("credit")

**COMPLETE:** write your own list of tickers. Keep it short (3 to 8 series): there are only about 40
training quarters.

`GDP_LAGS` adds past GDP growth as a predictor (0 = none). `FLATS = 3` uses the three months of the
quarter as separate columns.

In [ ]:
MY_VARS  = ["RCEM1100", "RVEH2200", "UNTL0000", "UGTR0000", "XBRT0001", "RENE5005"]   # COMPLETE
GDP_LAGS = 1
FLATS    = 3

Check the list. A ticket that is not in the cleaned data is dropped with a message. The table shows how
many months of the target quarter each series has at each vintage: at vintage -2, two months before the
quarter ends, series published late have none yet.

In [ ]:
missing = [v for v in MY_VARS if v not in catalog.index]
if missing:
    print(f"Not in the cleaned data, dropped: {missing}. Use find_series() to pick another ticket.")
MY_VARS = [v for v in MY_VARS if v in catalog.index]

check = catalog.loc[MY_VARS, ["description", "freq", "months_lag"]].copy()
for lag in lags:
    # months of the quarter already published at this vintage (0 to 3)
    check[f"months at {lag:+d}"] = (3 - (check.months_lag - lag)).clip(0, 3)
check

# Part 3: Your model

The nowcast loop goes over every test quarter and every vintage (-2 to +2). For each one it:

1. takes the training rows, every quarter before the test quarter;
2. builds the data as published at that vintage (`gen_lagged_data`) and fills the gaps with training means;
3. puts the months of each quarter side by side (`flatten_data`) and adds GDP lags (`lagged_target_v`);
4. **fits your model** and predicts the test quarter.

Step 4 is yours. The loop comes with OLS:

```python
model = LinearRegression()
model.fit(X_train, y_train)
```

**COMPLETE:** replace the first line with your method and its parameters, for example:

```python
model = make_pipeline(StandardScaler(), Ridge(alpha=10))
model = make_pipeline(StandardScaler(), Lasso(alpha=0.1))
model = make_pipeline(StandardScaler(), ElasticNet(alpha=0.1, l1_ratio=0.5))
model = RandomForestRegressor(n_estimators=300, max_depth=3, random_state=0)
model = GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, max_depth=2, random_state=0)
```

Penalized models (Ridge, Lasso, Elastic Net) need the columns on the same scale, hence
`make_pipeline(StandardScaler(), ...)`. Keep `model.fit(X_train, y_train)`: every scikit-learn model is
fitted the same way.

The frame with your variables and GDP, the test quarters and their actual GDP growth.

In [ ]:
frame = data[MY_VARS + [target_variable]].reset_index()
frame = frame.loc[(frame.date >= train_start_date) & (frame.date <= test_end_date)].reset_index(drop=True)

dates   = pd.date_range(test_start_date, test_end_date, freq="3MS").strftime("%Y-%m-%d").tolist()
actuals = frame.loc[frame.date.astype(str).isin(dates), target_variable].values
print(f"{len(dates)} test quarters: {dates[0]} to {dates[-1]}")

`make_X()` applies step 3 to a frame. The training rows and the quarter we predict go through exactly
the same steps.

In [ ]:
def make_X(df, lag):
    """Months of each quarter side by side, quarter rows only, GDP lags as published at `lag`."""
    out = flatten_data(df, target_variable, FLATS)
    out = quarter_rows(out)
    out = out.fillna(out.mean(numeric_only=True))
    return lagged_target_v(out, GDP_LAGS, metadata, lag)

The loop. Your model goes between the two `=====` lines.

In [ ]:
preds = {lag: [] for lag in lags}

for date in dates:
    # 1. training rows: every quarter before the test quarter
    train = frame.loc[frame.date <= pd.to_datetime(date) - pd.DateOffset(months=3)]
    train = mean_fill_dataset(train, train)

    for lag in lags:
        tr = make_X(train, lag)
        X_train = tr.drop(columns=["date", target_variable])
        y_train = tr[target_variable]

        # ===== YOUR MODEL: replace with your method and its parameters ==========
        model = LinearRegression()
        model.fit(X_train, y_train)
        # ==========================================================================

        # 2-3. the test quarter as published at this vintage, same steps as training
        now = gen_lagged_data(metadata, frame, date, lag)
        now = mean_fill_dataset(train, now)
        now = make_X(now, lag)
        X_now = now.loc[now.date == date, X_train.columns]

        # 4. the nowcast
        preds[lag].append(model.predict(X_now)[0])

print(f"Model: {model}")

# Part 4: Results

RMSE by vintage, next to the bar every model has to clear: always predicting the training-period mean
of GDP growth.

In [ ]:
results = pd.DataFrame(preds, index=pd.to_datetime(dates))
results.insert(0, "actual", actuals)

naive = frame.loc[frame.date < test_start_date, target_variable].mean()
naive_rmse = np.sqrt(np.mean((results["actual"] - naive) ** 2))

summary = pd.DataFrame({
    "vintage": lags,
    "RMSE": [np.sqrt(np.mean((results[lag] - results["actual"]) ** 2)) for lag in lags],
})
summary["vs training mean"] = summary["RMSE"] / naive_rmse   # below 1 = better than the mean
print(f"Training mean, RMSE: {naive_rmse:.4f}")
summary.round(3)

Actual GDP growth and your nowcast at vintage 0, the last month of the quarter.

In [ ]:
ax = results[["actual", 0]].rename(columns={0: "nowcast (vintage 0)"}).plot(
    figsize=(9, 4), style=["k-", "o--"])
ax.axhline(naive, color="grey", linestyle=":", label="training mean")
ax.set_ylabel("GDP growth, q/q %")
ax.legend()
plt.show()

## Nowcast of the current quarter

The same model, trained on every quarter with GDP, nowcasts `cfg.DESIRED_DATE` (2026Q3) at each
vintage. `clone()` copies your model with its parameters, unfitted. GDP for the quarters not published
yet is filled with the training mean so the rows are kept; with GDP published 6 months late, those
values are never used as predictors at these vintages.

In [ ]:
live = data[MY_VARS + [target_variable]].reset_index()
live = live.loc[(live.date >= train_start_date) & (live.date <= desired_date)].reset_index(drop=True)
train_all = mean_fill_dataset(live.loc[live.date <= test_end_date], live.loc[live.date <= test_end_date])

unpublished = live.date.dt.month.isin([3, 6, 9, 12]) & (live.date > test_end_date)
live.loc[unpublished, target_variable] = train_all[target_variable].mean()

current = {}
for lag in lags:
    tr = make_X(train_all, lag)
    X_train = tr.drop(columns=["date", target_variable])
    m = clone(model).fit(X_train, tr[target_variable])

    now = gen_lagged_data(metadata, live, desired_date, lag)
    now = mean_fill_dataset(train_all, now)
    now = make_X(now, lag)
    current[lag] = m.predict(now.loc[now.date == desired_date, X_train.columns])[0]

pd.Series(current, name=f"nowcast {desired_date:%Y-%m}").rename_axis("vintage").round(2)

## Save your result

Your vintage 0 RMSE goes on the workshop scoreboard (`output/workshop_results.csv`), next to the models
of Blocks 1 to 4. **COMPLETE:** give your model a name. A new name adds a row; the same name replaces it.

In [ ]:
MY_NAME = "OLS, my variables"   # COMPLETE

RESULTS_FILE = os.path.join(P["output"], "workshop_results.csv")
row = pd.DataFrame([{
    "block": 5, "activity": "act11", "language": "Python", "model": MY_NAME,
    "rmse_test": round(float(summary.loc[summary.vintage == 0, "RMSE"].iloc[0]), 4),
    "notes": "vintage 0; " + " ".join(MY_VARS),
}])
if os.path.exists(RESULTS_FILE) and os.path.getsize(RESULTS_FILE) > 0:
    prev = pd.read_csv(RESULTS_FILE)
    same = (prev["activity"] == "act11") & (prev["model"] == MY_NAME)
    row = pd.concat([prev[~same], row], ignore_index=True)
row.to_csv(RESULTS_FILE, index=False)

row.sort_values("rmse_test")

# Exercises

**Exercise 1.** Change `MY_VARS` (Part 2) and run the notebook again. Which series lower the RMSE at
vintage -2? And at vintage +2?

**Exercise 2.** Replace OLS with another method in the loop (Part 3). Try at least two settings of its
parameters and save each one under its own `MY_NAME`.

**Exercise 3.** Compare your best model with the training mean and with the models on the scoreboard.
Does it beat the mean at every vintage?